# 3. Politeness Feature Construction

*AI assistance (Claude-sonnet-4.6 in UvA AI Chat): notebook structure and ConvoKit pipeline setup.*

This notebook takes `messages.csv` produced by notebook 02 and extracts
politeness strategy scores for each message using ConvoKit's
`PolitenessStrategies` transformer (Danescu-Niculescu-Mizil et al., 2013).
It runs four steps in order:

| Step | Input | Output |
|------|-------|--------|
| 3.1 | `messages.csv` | Validated message dataframe |
| 3.2 | Message texts (via MediaWiki API) | `messages_with_text.csv` |
| 3.3 | `messages_with_text.csv` + ConvoKit | `politeness_scores.csv` |
| 3.4 | `politeness_scores.csv` | `politeness_profiles.csv` (newcomer-level) |

**Note:** `ConvoKit` is incompatible with Python 3.11 and above.

### 3.0 Configuration

In [1]:
import os
import time
import requests
import pandas as pd
from pathlib import Path
import convokit
from convokit import Corpus, Speaker, Utterance, PolitenessStrategies
from convokit.politenessStrategies import PolitenessStrategies
import spacy
from convokit import TextParser

In [2]:
# ── Working directory ────────────────────────────────────────────────────────
os.chdir("/Users/ada/bdacaII-2026-ada")

# ── API endpoint ──────────────────────────────────────────────────────────────
API = "https://en.wikipedia.org/w/api.php"

# ── Polite API call settings (must match notebooks 1 and 2) ──────────────────
MAXLAG              = 5    # seconds; cooperative DB-lag threshold
REQUEST_TIMEOUT     = 15   # seconds before a request is considered hung
RETRY_DELAY         = 2    # seconds to wait between retry attempts
MAX_RETRIES         = 5    # maximum retry attempts per API call
INTER_REQUEST_DELAY = 0.1  # polite pause between individual API calls (in seconds)
INTER_PAGE_DELAY    = 0.5  # polite pause between paginated API calls (in seconds)
SAMPLE_CAP = 1000          # max messages per sender type
SEED       = 42
BATCH_SIZE = 50            # MediaWiki API maximum revids per request
SAVE_EVERY = 20            # save progress every N batches

# ── Data directories (must match notebooks 1 and 2) ──────────────────────────
DATA_DIR      = Path("Final_project/data")
PROCESSED_DIR = DATA_DIR / "processed"

# ── Input paths ───────────────────────────────────────────────────────────────
MESSAGES_CSV  = PROCESSED_DIR / "messages.csv"

# ── Output paths ──────────────────────────────────────────────────────────────
MESSAGES_TEXT_CSV     = PROCESSED_DIR / "messages_with_text.csv"
POLITENESS_SCORES_CSV = PROCESSED_DIR / "politeness_scores.csv"
POLITENESS_PROFILES_CSV = PROCESSED_DIR / "politeness_profiles.csv"

# ── Module-level session (same User-Agent as notebooks 1 and 2) ──────────────
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "BDACA-II-course/1.0 (educational; adashiyj@gmail.com)"
})

print(f"pandas     {pd.__version__}")
print(f"Session ready — User-Agent: {SESSION.headers['User-Agent']}")

pandas     2.3.3
Session ready — User-Agent: BDACA-II-course/1.0 (educational; adashiyj@gmail.com)


### 3.1 Load, validate, and subsample message dataframe

Loads `messages.csv` from notebook 02 and performs two preparatory steps before any API calls are made:

1. **Validation**: checks that all expected columns are present and raises a `ValueError` immediately if not, rather than failing silently later in the pipeline.
2. **Subsampling**: sender types with more than `SAMPLE_CAP` messages are randomly subsampled to `SAMPLE_CAP` (seed = 42) before text is fetched. This is done here — before the API step — so the fetch and scoring steps only process the messages that will actually be used. The smallest sender type (Advisor, N = 270) is retained in full, yielding a worst-case margin of error of ±6.0% at the 95% confidence level.

Unidentified bots are excluded at this stage as they are not part of role-based analyses.

In [3]:
def load_and_subsample(
    path: Path = MESSAGES_CSV,
    cap:  int  = SAMPLE_CAP,
    seed: int  = SEED,
) -> pd.DataFrame:
    """Load, validate, and subsample the message-level dataframe.

    Parameters
    ----------
    path : Path to messages.csv produced by notebook 02.
    cap  : Maximum messages to retain per sender type. Sender types with
           fewer messages than cap are kept in full.
    seed : Random seed for reproducibility.

    Returns
    -------
    pd.DataFrame with columns: user_id, username, revid, timestamp,
    sender, sender_type, sizediff.

    Raises
    ------
    FileNotFoundError  If messages.csv does not exist.
    ValueError         If expected columns are missing.
    """
    if not path.exists():
        raise FileNotFoundError(
            f"messages.csv not found at {path}.\n"
            "Run notebook 02 first to generate it."
        )

    # ── Load ──────────────────────────────────────────────────────────────────
    df = pd.read_csv(path)

    # ── Validate ──────────────────────────────────────────────────────────────
    expected_cols = {"user_id", "username", "revid", "sender_type"}
    missing = expected_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing expected columns: {missing}")

    # ── Exclude unidentified bots ─────────────────────────────────────────────
    n_before = len(df)
    df       = df[df["sender_type"] != "unidentified_bot"].copy()
    n_excl   = n_before - len(df)

    print(f"Messages loaded        : {n_before:,}")
    print(f"Unidentified bots excl : {n_excl:,}")
    print(f"Messages for scoring   : {len(df):,}")
    print(f"\nSender type counts (before subsampling):")
    print(df["sender_type"].value_counts().to_string())

    # ── Subsample ─────────────────────────────────────────────────────────────
    df_sampled = (
        df
        .groupby("sender_type", group_keys=False)
        .apply(lambda g: g.sample(min(len(g), cap), random_state=seed))
        .reset_index(drop=True)
    )

    print(f"\nSender type counts (after subsampling to {cap:,} per type):")
    print(df_sampled["sender_type"].value_counts().to_string())
    print(f"\nTotal messages to fetch and score: {len(df_sampled):,}")

    return df_sampled



df_messages_raw = load_and_subsample()
df_messages_raw.head()

Messages loaded        : 479,344
Unidentified bots excl : 626
Messages for scoring   : 478,718

Sender type counts (before subsampling):
sender_type
human       391244
Clerk        61717
Tagger       11075
Notifier      7620
Fixer         6792
Advisor        270

Sender type counts (after subsampling to 1,000 per type):
sender_type
Clerk       1000
Fixer       1000
Tagger      1000
Notifier    1000
human       1000
Advisor      270

Total messages to fetch and score: 5,270


/var/folders/kw/83vtrb5s4k5g0zx9zp814ky00000gn/T/ipykernel_91274/3553119727.py:55: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), cap), random_state=seed))


,user_id,username,revid,timestamp,sender,sender_type,sizediff
0,170080587,JJMC89,708686313,2016-03-07T00:23:39Z,ReferenceBot,Advisor,8503
1,170049730,JJMC89,733043738,2016-08-05T00:22:40Z,ReferenceBot,Advisor,12336
2,170049784,JJMC89,748394775,2016-11-08T00:25:37Z,ReferenceBot,Advisor,8890
3,170049716,JJMC89,734989791,2016-08-18T00:24:32Z,ReferenceBot,Advisor,7333
4,170049581,JJMC89,733043738,2016-08-05T00:22:40Z,ReferenceBot,Advisor,12336


### 3.2 Fetch message text from the MediaWiki API (batched)

Retrieves the wikitext content of each revision using the `revisions` endpoint with `rvprop=content`. Revisions are fetched in **batches of 50** (the API maximum per request) rather than one at a time. This reduces the total number of HTTP requests by up to 50×, substantially cutting collection time.

Progress is saved every `SAVE_EVERY` batches so the step can be safely interrupted and resumed. On resume, already-fetched revision IDs are skipped.

Empty text (returned for deleted or suppressed revisions) is recorded as an empty string and flagged in Step 3.3 rather than being silently dropped, so the proportion of missing text per sender type can be reported as a data quality constraint.

In [4]:
def fetch_batch_texts(revids: list[int]) -> dict[int, str]:
    """Fetch wikitext for a batch of up to 50 revision IDs in one API call.

    Parameters
    ----------
    revids : List of revision IDs (max 50).

    Returns
    -------
    Dict mapping revid -> wikitext content string (empty string on failure).
    """
    params = {
        "action":        "query",
        "prop":          "revisions",
        "revids":        "|".join(str(r) for r in revids),
        "rvprop":        "ids|content",   # ids required to get revid in response
        "rvslots":       "main",
        "format":        "json",
        "formatversion": "2",
        "maxlag":        str(MAXLAG),
    }

    for attempt in range(MAX_RETRIES):
        try:
            resp = SESSION.get(API, params=params, timeout=REQUEST_TIMEOUT)
            data = resp.json()

            # Honour maxlag retry signal from the server.
            if "error" in data and data["error"].get("code") == "maxlag":
                wait = int(resp.headers.get("Retry-After", RETRY_DELAY))
                time.sleep(wait)
                continue

            pages  = data.get("query", {}).get("pages", [])
            result = {}
            for page in pages:
                for rev in page.get("revisions", []):
                    rid     = rev.get("revid")
                    content = rev.get("slots", {}).get("main", {}).get("content", "")
                    if rid is not None:
                        result[rid] = content
            return result

        except Exception:
            time.sleep(RETRY_DELAY * (attempt + 1))

    # All retries exhausted, return empty strings for the whole batch.
    return {r: "" for r in revids}


def collect_message_texts(
    df: pd.DataFrame,
    output_path: Path = MESSAGES_TEXT_CSV,
    batch_size:  int  = BATCH_SIZE,
    save_every:  int  = SAVE_EVERY,
) -> pd.DataFrame:
    """Fetch revision text for all messages in df using batched API calls.

    Skips revisions already fetched if output_path exists (crash-safe resume).

    Parameters
    ----------
    df          : Subsampled message dataframe from load_and_subsample().
    output_path : CSV path for incremental saves.
    batch_size  : Number of revids per API request (max 50).
    save_every  : Save progress every N batches.

    Returns
    -------
    df with a new column `text` containing the revision wikitext.
    """
    # ── Resume from prior run if available ────────────────────────────────────
    if output_path.exists():
        df_saved     = pd.read_csv(output_path)
        already_done = set(df_saved["revid"])
        print(f"Resuming: {len(already_done):,} revisions already fetched.")
    else:
        df_saved     = pd.DataFrame(columns=["revid", "text"])
        already_done = set()

    to_fetch  = df[~df["revid"].isin(already_done)]["revid"].tolist()
    n_total   = len(to_fetch)
    n_batches = (n_total + batch_size - 1) // batch_size
    print(f"Fetching {n_total:,} revisions in {n_batches:,} batches of {batch_size}.")

    results = []
    for batch_idx in range(n_batches):
        batch = to_fetch[batch_idx * batch_size : (batch_idx + 1) * batch_size]
        texts = fetch_batch_texts(batch)
        for revid in batch:
            results.append({"revid": revid, "text": texts.get(revid, "")})

        # ── Incremental save ──────────────────────────────────────────────────
        if (batch_idx + 1) % save_every == 0:
            df_partial  = pd.DataFrame(results)
            df_combined = pd.concat([df_saved, df_partial], ignore_index=True)
            df_combined.to_csv(output_path, index=False)
            pct = (batch_idx + 1) / n_batches * 100
            print(f"  Batch {batch_idx + 1}/{n_batches} ({pct:.0f}%) — saved.")

        time.sleep(INTER_REQUEST_DELAY)  # polite pause between batches

    # ── Final save ────────────────────────────────────────────────────────────
    df_new = pd.DataFrame(results)
    df_all = pd.concat([df_saved, df_new], ignore_index=True).drop_duplicates("revid")
    df_all.to_csv(output_path, index=False)

    # ── Merge text back onto message dataframe ────────────────────────────────
    df_with_text         = df.merge(df_all[["revid", "text"]], on="revid", how="left")
    df_with_text["text"] = df_with_text["text"].fillna("")

    empty = (df_with_text["text"] == "").sum()
    print(f"\nFetch complete.")
    print(f"  Total messages : {len(df_with_text):,}")
    print(f"  Empty text     : {empty:,} ({empty / len(df_with_text):.1%})")

    return df_with_text


df_messages_text = collect_message_texts(df_messages_raw)
df_messages_text.head()

Fetching 5,270 revisions in 106 batches of 50.
  Batch 20/106 (19%) — saved.
  Batch 40/106 (38%) — saved.
  Batch 60/106 (57%) — saved.
  Batch 80/106 (75%) — saved.
  Batch 100/106 (94%) — saved.

Fetch complete.
  Total messages : 5,270
  Empty text     : 24 (0.5%)


,user_id,username,revid,timestamp,sender,sender_type,sizediff,text
0,170080587,JJMC89,708686313,2016-03-07T00:23:39Z,ReferenceBot,Advisor,8503,{{/header}}\n{{User:MiszaBot/config\n| algo ...
1,170049730,JJMC89,733043738,2016-08-05T00:22:40Z,ReferenceBot,Advisor,12336,{{/header}}\n{{User:MiszaBot/config\n| algo ...
2,170049784,JJMC89,748394775,2016-11-08T00:25:37Z,ReferenceBot,Advisor,8890,{{/header}}\n{{User:MiszaBot/config\n| algo ...
3,170049716,JJMC89,734989791,2016-08-18T00:24:32Z,ReferenceBot,Advisor,7333,{{/header}}\n{{User:MiszaBot/config\n| algo ...
4,170049581,JJMC89,733043738,2016-08-05T00:22:40Z,ReferenceBot,Advisor,12336,{{/header}}\n{{User:MiszaBot/config\n| algo ...


### 3.3 Compute politeness strategies with ConvoKit

Each message is passed through two ConvoKit transformers in sequence:

1. **`TextParser`** — generates spaCy dependency parses required by the politeness classifier.
2. **`PolitenessStrategies`** (Danescu-Niculescu-Mizil et al., 2013) — yields 21 binary strategy scores per message.

Messages with empty text (fetch failures from Step 3.2) are assigned zero scores for all strategies and flagged with `scored=False`. They are excluded from averaging in Step 3.4 but retained in the output file so the number of missing observations per sender type is visible.

**Note:** This step requires the `convokit_env` Python 3.10 environment. ConvoKit is incompatible with Python 3.11 and above.

In [5]:
def build_convokit_corpus(df: pd.DataFrame) -> Corpus:
    """Convert the message dataframe into a ConvoKit Corpus.

    Only non-empty messages are added; empty-text rows are handled
    separately after scoring.

    Parameters
    ----------
    df : Message dataframe with columns revid, sender, text.

    Returns
    -------
    ConvoKit Corpus ready for TextParser + PolitenessStrategies.
    """
    # NaN-safe filter: keep only rows with real text
    mask        = df["text"].notna() & (df["text"].str.strip() != "")
    df_nonempty = df[mask].copy()

    print(f"  Rows with scoreable text : {len(df_nonempty):,}")
    print(f"  Rows with NaN/empty text : {(~mask).sum():,}")

    if df_nonempty.empty:
        raise ValueError(
            "No non-empty text rows found in df. "
            "Check that Step 3.2 ran correctly and produced df_messages_text."
        )

    speakers = {
        str(sender): Speaker(id=str(sender))
        for sender in df_nonempty["sender"].unique()
    }

    utterances = [
        Utterance(
            id=str(row.revid),
            speaker=speakers[str(row.sender)],
            conversation_id="all_messages",
            text=str(row.text),
        )
        for row in df_nonempty.itertuples(index=False)
    ]

    print(f"  Corpus built: {len(utterances):,} utterances, "
          f"{len(speakers):,} unique speakers.")
    return Corpus(utterances=utterances)


def score_politeness(
    df: pd.DataFrame,
    output_path: Path = POLITENESS_SCORES_CSV,
) -> pd.DataFrame:
    """Extract politeness strategy scores for all messages.

    Pipeline:
      1. Build ConvoKit Corpus from non-empty messages.
      2. Parse utterances with TextParser (spaCy dependency parse).
      3. Score with PolitenessStrategies → 21 binary strategy columns.
      4. Append zero-score rows for empty/NaN-text messages (scored=False).

    Parameters
    ----------
    df          : Text-enriched message dataframe from collect_message_texts().
    output_path : CSV path to save scores to.

    Returns
    -------
    pd.DataFrame with revid, user_id, sender_type, scored,
    and 21 politeness strategy columns.
    """
    if output_path.exists():
        print(f"Scores already exist at {output_path} — loading.")
        return pd.read_csv(output_path)

    # ── Step 1: build corpus ──────────────────────────────────────────────────
    print("Building ConvoKit corpus ...")
    corpus = build_convokit_corpus(df)

    # ── Step 2: parse with spaCy ──────────────────────────────────────────────
    print("Parsing utterances with TextParser (spaCy) ...")
    parser = TextParser(verbosity=100)
    corpus = parser.transform(corpus)

    # ── Step 3: score politeness strategies ───────────────────────────────────
    print("Scoring politeness strategies ...")
    ps     = PolitenessStrategies()
    corpus = ps.transform(corpus, markers=True)

    # ── Step 4: extract scores ────────────────────────────────────────────────
    rows = []
    for utt in corpus.iter_utterances():
        scores = utt.retrieve_meta("politeness_strategies")
        if scores is None:
            continue
        try:
            revid = int(utt.id)
        except (ValueError, TypeError):
            print(f"  Warning: could not parse utt.id={utt.id!r} as int — skipping.")
            continue
        row = {"revid": revid}
        row.update(scores)
        rows.append(row)

    print(f"  Utterances with scores extracted: {len(rows):,}")

    if len(rows) == 0:
        sample_ids  = [utt.id for utt in list(corpus.iter_utterances())[:5]]
        sample_meta = [list(utt.meta.keys()) for utt in list(corpus.iter_utterances())[:3]]
        print(f"  Sample utt.id values : {sample_ids}")
        print(f"  Sample meta keys     : {sample_meta}")
        raise RuntimeError(
            "No politeness scores extracted. "
            "Check sample utt.id values and meta keys printed above."
        )

    df_scores = pd.DataFrame(rows)
    print(f"  df_scores columns: {df_scores.columns.tolist()}")

    # ── Step 5: merge user_id and sender_type back in ─────────────────────────
    df_meta   = df[["revid", "user_id", "sender_type"]].drop_duplicates(subset="revid")
    df_scores = df_scores.merge(df_meta, on="revid", how="left")
    df_scores["scored"] = True

    # ── Step 6: append zero-score rows for empty/NaN-text messages ───────────
    strategy_cols = [
        c for c in df_scores.columns
        if c not in {"revid", "user_id", "sender_type", "scored"}
    ]
    empty_mask = df["text"].isna() | (df["text"].str.strip() == "")
    empty_rows = df[empty_mask][["revid", "user_id", "sender_type"]].copy()

    if not empty_rows.empty:
        for col in strategy_cols:
            empty_rows[col] = 0
        empty_rows["scored"] = False
        df_scores = pd.concat([df_scores, empty_rows], ignore_index=True)
        print(f"  Empty/NaN-text messages (zero-scored): {len(empty_rows):,}")

    df_scores.to_csv(output_path, index=False)

    n_scored   = df_scores["scored"].sum()
    n_unscored = (~df_scores["scored"]).sum()
    print(f"\nPoliteness scoring complete.")
    print(f"  Messages scored      : {n_scored:,}")
    print(f"  Messages zero-scored : {n_unscored:,}")
    print(f"  Strategies extracted : {len(strategy_cols)}")

    return df_scores

df_scores = score_politeness(df_messages_text)
df_scores.head()

Building ConvoKit corpus ...
  Rows with scoreable text : 5,246
  Rows with NaN/empty text : 24
  Corpus built: 5,246 utterances, 497 unique speakers.
Parsing utterances with TextParser (spaCy) ...
100/2989 utterances processed
200/2989 utterances processed
300/2989 utterances processed
400/2989 utterances processed
500/2989 utterances processed
600/2989 utterances processed
700/2989 utterances processed
800/2989 utterances processed
900/2989 utterances processed
1000/2989 utterances processed
1100/2989 utterances processed
1200/2989 utterances processed
1300/2989 utterances processed
1400/2989 utterances processed
1500/2989 utterances processed
1600/2989 utterances processed
1700/2989 utterances processed
1800/2989 utterances processed
1900/2989 utterances processed
2000/2989 utterances processed
2100/2989 utterances processed
2200/2989 utterances processed
2300/2989 utterances processed
2400/2989 utterances processed
2500/2989 utterances processed
2600/2989 utterances processed
2700/

,revid,feature_politeness_==Please==,feature_politeness_==Please_start==,feature_politeness_==HASHEDGE==,feature_politeness_==Indirect_(btw)==,feature_politeness_==Hedges==,feature_politeness_==Factuality==,feature_politeness_==Deference==,feature_politeness_==Gratitude==,feature_politeness_==Apologizing==,...,feature_politeness_==Indirect_(greeting)==,feature_politeness_==Direct_question==,feature_politeness_==Direct_start==,feature_politeness_==HASPOSITIVE==,feature_politeness_==HASNEGATIVE==,feature_politeness_==SUBJUNCTIVE==,feature_politeness_==INDICATIVE==,user_id,sender_type,scored
0,708686313,1,1,1,0,1,1,0,1,1,...,1,1,1,1,1,0,1,170080587,Advisor,True
1,733043738,1,1,1,0,1,1,0,1,1,...,1,1,1,1,1,0,1,170049730,Advisor,True
2,748394775,1,1,1,0,1,1,0,1,1,...,1,0,1,1,1,0,0,170049784,Advisor,True
3,734989791,1,1,1,0,1,1,0,1,0,...,1,1,0,1,1,0,0,170049716,Advisor,True
4,737617758,1,1,1,0,1,1,0,1,1,...,1,1,1,1,1,1,1,170049666,Advisor,True


### 3.4 Aggregate politeness scores to newcomer level

Averages the 21 strategy scores per sender type per newcomer, using only successfully scored messages (`scored=True`). Column names follow the pattern `{sender_type}__{strategy_name}` to make the sender type explicit in the analysis dataset.

Newcomers who received no messages from a given sender type (or whose messages all failed to score) receive a value of 0 for that sender type's strategy columns. This matches the contact dummy convention used in notebook 02 and avoids introducing NaN values into the analysis dataset.

In [6]:
def build_politeness_profiles(
    df_scores: pd.DataFrame,
    output_path: Path = POLITENESS_PROFILES_CSV,
) -> pd.DataFrame:
    """Aggregate message-level politeness scores to the newcomer level.

    For each newcomer and sender type, computes the mean score for each
    of the 20 politeness strategies across all messages received from
    that sender type. Column names follow the pattern:
    {sender_type}__{strategy_name}.

    Parameters
    ----------
    df_scores   : Message-level politeness scores (from Step 3.3).
    output_path : CSV path to save the newcomer-level profiles to.

    Returns
    -------
    pd.DataFrame with one row per newcomer and aggregated strategy columns.
    """
    strategy_cols = [c for c in df_scores.columns
                     if c not in {"revid", "user_id", "sender_type", "scored"}]

    # Only use successfully scored messages for averaging.
    df_scored = df_scores[df_scores["scored"] == True].copy()

    sender_types = df_scored["sender_type"].unique()
    profile_dfs  = []

    for sender_type in sender_types:
        df_sender = df_scored[df_scored["sender_type"] == sender_type]

        # Average strategy scores per newcomer for this sender type.
        df_agg = (
            df_sender.groupby("user_id")[strategy_cols]
            .mean()
            .reset_index()
        )

        # Prefix column names with sender type for clarity.
        rename_map = {
            col: f"{sender_type}__{col}" for col in strategy_cols
        }
        df_agg = df_agg.rename(columns=rename_map)
        profile_dfs.append(df_agg)

    # Merge all sender-type profiles on user_id.
    df_profiles = profile_dfs[0]
    for df_next in profile_dfs[1:]:
        df_profiles = df_profiles.merge(df_next, on="user_id", how="outer")

    # Fill NaN with 0 — newcomer received no messages from that sender type.
    df_profiles = df_profiles.fillna(0)

    df_profiles.to_csv(output_path, index=False)
    print(f"Politeness profiles: {len(df_profiles):,} newcomers x "
          f"{len(df_profiles.columns)} columns")
    print(f"Saved to {output_path}")

    return df_profiles


df_profiles = build_politeness_profiles(df_scores)
df_profiles.head()

Politeness profiles: 113 newcomers x 127 columns
Saved to Final_project/data/processed/politeness_profiles.csv


,user_id,Advisor__feature_politeness_==Please==,Advisor__feature_politeness_==Please_start==,Advisor__feature_politeness_==HASHEDGE==,Advisor__feature_politeness_==Indirect_(btw)==,Advisor__feature_politeness_==Hedges==,Advisor__feature_politeness_==Factuality==,Advisor__feature_politeness_==Deference==,Advisor__feature_politeness_==Gratitude==,Advisor__feature_politeness_==Apologizing==,...,human__feature_politeness_==1st_person_start==,human__feature_politeness_==2nd_person==,human__feature_politeness_==2nd_person_start==,human__feature_politeness_==Indirect_(greeting)==,human__feature_politeness_==Direct_question==,human__feature_politeness_==Direct_start==,human__feature_politeness_==HASPOSITIVE==,human__feature_politeness_==HASNEGATIVE==,human__feature_politeness_==SUBJUNCTIVE==,human__feature_politeness_==INDICATIVE==
0,170000586,1.0,1.0,1.0,0.5,1.0,0.5,0.5,1.0,1.0,...,1.000000,1.0,1.000000,0.625000,0.375000,0.750000,1.0,1.0,0.250000,0.625000
1,170000636,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.833333,1.0,0.666667,0.500000,0.333333,0.833333,1.0,1.0,0.500000,0.666667
2,170000705,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,...,1.000000,1.0,1.000000,0.666667,0.333333,0.555556,1.0,1.0,0.444444,0.555556
3,170000714,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,...,1.000000,1.0,0.833333,0.500000,0.500000,0.666667,1.0,1.0,0.000000,0.666667
4,170000725,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.0,1.0,0.500000,0.500000


### 3.5 Summary

Sanity check confirming all three output files were written correctly before proceeding to analysis. If `politeness_profiles.csv` shows `NOT FOUND`, re-run Steps 3.3 and 3.4. If `messages_with_text.csv` is missing, start from Step 3.2.

In [7]:
print("=" * 55)
print("Politeness feature construction complete")
print("=" * 55)

for path in (MESSAGES_TEXT_CSV, POLITENESS_SCORES_CSV, POLITENESS_PROFILES_CSV):
    if path.exists():
        df_check = pd.read_csv(path)
        print(f"  {path.name:40s} {len(df_check):>6,} rows x {len(df_check.columns)} cols")
    else:
        print(f"  {path.name:40s} NOT FOUND")

print()

Politeness feature construction complete
  messages_with_text.csv                    2,998 rows x 2 cols
  politeness_scores.csv                     3,013 rows x 25 cols
  politeness_profiles.csv                     113 rows x 127 cols

